In [2]:
!pip install --ignore-installed blinker

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)


In [3]:
!pip install mlflow dagshub optuna imbalanced-learn

  Using cached mlflow-3.11.1-py3-none-any.whl.metadata (49 kB)
  Using cached dagshub-0.6.9-py3-none-any.whl.metadata (12 kB)
  Using cached optuna-4.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached mlflow_skinny-3.11.1-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_tracing-3.11.1-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.6-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached sqlalchemy-2.0.49-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.meta

In [4]:
#!aws configure
import dagshub
dagshub.init(repo_owner="rohitbedse",repo_name="yt-comment-sentiment-analysis",mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f5176c17-50bb-4532-b842-e6d2f91800f2&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3fe352479ab1c0e6c0399319518707bee4317f89a0568f9b8cf78938bc475ed3




Accessing as rohitbedse

Initialized MLflow to track repo "rohitbedse/yt-comment-sentiment-analysis"

Repository rohitbedse/yt-comment-sentiment-analysis initialized!

In [5]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow")

In [6]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning With Chnaging Features")

<Experiment: artifact_location='mlflow-artifacts:/54b3eaefe0cb4e4d8246724361152697', creation_time=1775572221042, experiment_id='9', last_update_time=1775572221042, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning With Chnaging Features', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report,f1_score
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
import numpy as np

In [11]:
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [12]:
# -----------------------------
# STEP 1: Label Mapping
# -----------------------------
df = df.dropna(subset=['category'])
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# -----------------------------
# STEP 2: Show ORIGINAL label distribution
# -----------------------------
print("🔥 ORIGINAL LABELS:")
print(np.unique(df['category'], return_counts=True))

# -----------------------------
# STEP 3: TRAIN-TEST SPLIT (FIRST)
# -----------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# -----------------------------
# STEP 4: TF-IDF (FIT ONLY ON TRAIN)
# -----------------------------
vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=5000)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# -----------------------------
# STEP 5: Show TRAIN distribution BEFORE SMOTE
# -----------------------------
print("\n📊 TRAIN LABELS BEFORE SMOTE:")
print(np.unique(y_train, return_counts=True))

# -----------------------------
# STEP 6: APPLY SMOTE ONLY ON TRAIN
# -----------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# -----------------------------
# STEP 7: Show TRAIN distribution AFTER SMOTE
# -----------------------------
print("\n🚀 TRAIN LABELS AFTER SMOTE:")
print(np.unique(y_train_res, return_counts=True))

# -----------------------------
# STEP 9: MLflow logging
# -----------------------------
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Bigram")
        mlflow.set_tag("experiment", "No_Leakage_Pipeline")

        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("resampling", "SMOTE")
        mlflow.log_param("vectorizer", "TF-IDF Bigram")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(model, f"{model_name}_model")
# ================================
# Step 6: Optuna Objective (FIXED KNN)
# ================================
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 3, 30)
    p = trial.suggest_categorical('p', [1, 2])

    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        p=p
    )

    # ✅ SAME as other models → split train into train/val
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_res, y_train_res,
        test_size=0.2,
        random_state=42,
        stratify=y_train_res
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    return f1_score(y_val, y_pred, average='macro')


# ================================
# Step 7: Run Optuna
# ================================
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_knn, n_trials=30)

    print("\n🏆 BEST PARAMS:", study.best_params)

    best_model = KNeighborsClassifier(
        n_neighbors=study.best_params['n_neighbors'],
        p=study.best_params['p']
    )

    # ✅ SAME logging as others
    log_mlflow("KNN", best_model, X_train_res, X_test, y_train_res, y_test)


# ================================
# RUN
# ================================
run_optuna_experiment()

🔥 ORIGINAL LABELS:
(array([0, 1, 2]), array([12644, 15770,  8248]))

📊 TRAIN LABELS BEFORE SMOTE:
(array([0, 1, 2]), array([10115, 12616,  6598]))


[I 2026-04-07 18:40:26,343] A new study created in memory with name: no-name-0dec74a1-6740-49b6-b49e-f794b14038d6



🚀 TRAIN LABELS AFTER SMOTE:
(array([0, 1, 2]), array([12616, 12616, 12616]))


[I 2026-04-07 18:40:28,789] Trial 0 finished with value: 0.37228520486854944 and parameters: {'n_neighbors': 27, 'p': 2}. Best is trial 0 with value: 0.37228520486854944.
[I 2026-04-07 18:40:31,197] Trial 1 finished with value: 0.3662951621219069 and parameters: {'n_neighbors': 30, 'p': 2}. Best is trial 0 with value: 0.37228520486854944.
[I 2026-04-07 18:40:34,534] Trial 2 finished with value: 0.2008536472027085 and parameters: {'n_neighbors': 13, 'p': 1}. Best is trial 0 with value: 0.37228520486854944.
[I 2026-04-07 18:40:36,967] Trial 3 finished with value: 0.39634774640818343 and parameters: {'n_neighbors': 18, 'p': 2}. Best is trial 3 with value: 0.39634774640818343.
[I 2026-04-07 18:40:40,882] Trial 4 finished with value: 0.2117680600489306 and parameters: {'n_neighbors': 9, 'p': 1}. Best is trial 3 with value: 0.39634774640818343.
[I 2026-04-07 18:40:43,284] Trial 5 finished with value: 0.4054453849384507 and parameters: {'n_neighbors': 15, 'p': 2}. Best is trial 5 with value: 


🏆 BEST PARAMS: {'n_neighbors': 3, 'p': 2}


2026/04/07 18:42:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 18:42:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run KNN_SMOTE_TFIDF_Bigram at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9/runs/86893b381d964e4ab3d1f1eb1d4abf6e
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9


In [13]:
# ✅ Best params from Optuna (example)
best_params = {
    'n_neighbors': 3,
    'p': 2
}

# ✅ Create KNN model
model = KNeighborsClassifier(
    n_neighbors=best_params['n_neighbors'],
    p=best_params['p']
)

# ✅ Train on SMOTE data
model.fit(X_train_res, y_train_res)

# ✅ Predict on test
y_pred = model.predict(X_test)

# ✅ Metrics
from sklearn.metrics import f1_score

print("Test Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Test Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

Test Macro F1: 0.42135849604123554
Test Weighted F1: 0.39820638127869823


In [14]:
import numpy as np

print("Unique predictions:", np.unique(y_pred))
print("Unique true labels:", np.unique(y_test))

Unique predictions: [0 1 2]
Unique true labels: [0 1 2]
